# Bootstrap Coverage Study

This notebook evaluates pointwise bootstrap coverage for fitted background models in the high-mass diphoton setting. Each toy dataset is generated from a fitted reference model; bootstrap resamples are refit, and percentile intervals are compared with the known generating PDF.

## Roadmap

1. Set the number of generated toys, bootstrap resamples, and evaluation points.
2. Load the data and fit the candidate truth models.
3. Submit or run the bootstrap fits, which are cached under `cache/coverage/`.
4. Aggregate the cached intervals and plot the empirical coverage as a function of diphoton mass.

The configured nominal coverage is $1 - \alpha$; failed fit attempts are omitted rather than treated as predictions.

In [ ]:
import matplotlib.pyplot as plt

import numpy as np
import ROOT

from emm import coverage as cov
from emm import data
from emm import models

In [ ]:
# Configuration
n_toys = 400
n_bootstraps = 1000
n_events_per_toy = 5036

# Test Config
# n_toys = 2
# n_bootstraps = 2
# n_events_per_toy = 5036

np.random.seed(42)
seeds = np.random.randint(0, 10000, size=n_toys)

n_test_points = 1000
test_range = (500, 3500)
test_points = np.linspace(test_range[0], test_range[1], n_test_points)

In [ ]:
# Load the data
data_tree = data.get_diphoton_data(tree=True)
x = ROOT.RooRealVar("x", "Diphoton Mass [GeV]", 500, 10_000)
data = ROOT.RooDataSet("mgg", "mgg", ROOT.RooArgSet(x), ROOT.RooFit.Import(data_tree))
n = data.numEntries()

In [ ]:
# Set up models
toy_models = [
    models.f1(x, prefix="true"),
    models.f2(x, prefix="true"),
    models.f3(x, prefix="true"),
    models.f4(x, prefix="true"),
]

model_primitives = [
    models.ModelPrimitive(models.f1),
    models.ModelPrimitive(models.f2),
    models.ModelPrimitive(models.f3),
    models.ModelPrimitive(models.f4),
]
for k in [2, 3, 4]:
    model_primitives.append(
        models.ModelPrimitive(
            models.ExponentialMixtureModel,
            k,
            data_mean=data.mean(x),
            name=f"ExponentialMixture-{k}",
        )
    )


for toy_model in toy_models:
    print(f"Fitting toy model: {toy_model.name}")
    toy_model.pdf.fitTo(data)

true_pdf_vals = {
    toy_model.name: models.evaluate_pdf(x, toy_model, test_points)
    for toy_model in toy_models
}

## Bootstrap Fits and Coverage Aggregation

For each seed and truth model, the execution cell generates one toy dataset and repeatedly bootstraps it. Each bootstrap fit stores its predicted PDF values at `test_points`; the aggregation step forms percentile intervals and checks whether the fitted truth PDF lies inside them. Uncomment the run call only when regenerating `cache/coverage/` results.

In [ ]:
# Run jobs
# cov.run_coverage_tasks(
#     x, toy_models, model_primitives,
#     seeds, n_bootstraps,
#     n_events_per_toy, test_points,
#     use_condor=True,
#     # remake=True,
# )

In [ ]:
coverages = cov.get_coverage_results(
    toy_models,
    seeds,
    n_bootstraps,
    n_events_per_toy,
    true_pdf_vals,
    alpha=0.05,
)

## Interpreting the Coverage Curves

The final plot reports the fraction of pseudo-experiments whose bootstrap interval contains the generating PDF at each mass point. Curves near the horizontal nominal-coverage level indicate calibrated intervals; systematic departures show where the fitting and resampling procedure under- or over-covers.

In [ ]:
# Plot the results
cov.plot_coverages(
    coverages, test_points,
    alpha=0.05, y_min=0.8,
    range=(500,3500),
    skip_every=20
)